<a href="https://colab.research.google.com/github/Avinashbitm/AI_learning/blob/main/session3_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background:linear-gradient(135deg,#10131c,#1f2540 60%,#3b1d5e);color:#fff;border-radius:14px;padding:26px 30px">
<img src="https://trendytech.in/wp-content/uploads/2021/05/trendytech_new.png" height="58" style="background:#fff;border-radius:8px;padding:4px 8px">
<div style="font-size:13px;font-weight:700;letter-spacing:.14em;color:#ff9a5c;margin-top:16px">AGENTIC AI ENGINEERING / FDE · WEEK 2</div>
<div style="font-size:34px;font-weight:800;line-height:1.15;margin:6px 0">Retrieval Augmented Generation (RAG)</div>
<div style="color:#c5cad6;font-size:15px">Give the LLM your company's knowledge: chunk, embed, store, retrieve, generate.</div>
</div>

**Running example:** ShopEasy's customer support assistant, answering from ShopEasy's policy handbook.

**Keys needed** (Colab → 🔑 Secrets, notebook access on): `GROQ_API_KEY`, `LANGSMITH_API_KEY`

<span style="color:#6b7385;font-size:12px">© TrendyTech. All rights reserved. Do not copy or share.</span>

---
## 0 · Setup
Turn on LangSmith tracing **first**, before any LangChain call, so every step is recorded.

In [ ]:
!pip install -q langchain langchain-groq langchain-text-splitters langchain-huggingface sentence-transformers

import os
from google.colab import userdata
from IPython.display import Markdown

In [25]:
# Turn tracing on BEFORE any LLM object exists.

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["AI_lang_smith_key"] = userdata.get("AI_lang_smith_key")
os.environ["LANGSMITH_PROJECT"] = "shopeasy-rag1"
os.environ["GROQ_API_KEY"] = userdata.get("grq_tredence")

In [29]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")

---
## 1 · The problem: the LLM doesn't know your company

In [30]:
response = llm.invoke([
    ("system", "You are ShopEasy's customer support assistant. Answer briefly."),
    ("user", "I bought a dress 12 days ago. Can I still return it?"),
])


display(Markdown(response.content))

Yes—you’re still within our typical 30‑day return window. Just make sure the dress is unworn, has its original tags, and you have the receipt. You can start the return online or bring it to any Store Easy location. If you need any help, let us know!

In [31]:
response = llm.invoke([
    ("system", "You are ShopEasy's customer support assistant. Answer briefly."),
    ("user", "What is ShopEasy's customer care phone number?"),
])

display(Markdown(response.content))

ShopEasy’s customer care can be reached at **1‑800‑555‑1234**.

> **Takeaway:** The model has never seen ShopEasy's policies, so it guesses: an invented 30-day return policy, and on some runs an invented phone number (on others it admits it doesn't know). The real number, 1800-000-4321, is in our handbook. Run both cells again and watch the answers change.

### Will a bigger model fix this?
Same question, three OpenAI models, from small to flagship.

In [32]:
# may it is not running because it is a paid model

In [33]:
!pip install -q langchain-openai

from langchain_openai import ChatOpenAI

os.environ["AGENTIC_OPEN_AI_KEY"] = userdata.get("AGENTIC_OPEN_AI_KEY")

questions = [
    "What is ShopEasy's customer care phone number?",
    "I bought a dress 12 days ago. Can I still return it?",
]

for model in ["gpt-5-mini", "gpt-5.4-mini", "gpt-5.5"]:
    display(Markdown(f"### {model}"))
    llm_openai = ChatOpenAI(model=model)

    for question in questions:
        response = llm_openai.invoke([
            ("system", "You are ShopEasy's customer support assistant. Answer briefly."),
            ("user", question),
        ])
        display(Markdown(f"**{question}**\n\n{response.content}"))

### gpt-5-mini

OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: lsv2_pt_***************************************6022. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

---
## 2 · Our knowledge: the ShopEasy policy handbook
A 25-section handbook, about 2,500 tokens.

In [ ]:
%%writefile shopeasy_handbook.txt
ShopEasy Customer Policy Handbook
Version 3.2, effective 1 October 2026

1. About ShopEasy
ShopEasy is an online shopping platform for fashion, home, beauty and electronics. We deliver to more than 19,000 pin codes across India. This handbook explains how orders, payments, deliveries, returns, refunds and warranties work. If anything in this handbook is unclear, our customer care team is happy to help.

2. Placing an Order
You can place an order on the ShopEasy website or the ShopEasy app. You need a registered account with a verified mobile number. Once an order is placed, you will receive an order confirmation by SMS, email and app notification. Each order has a unique order ID, for example SE-4521-889. Please keep this ID handy when you contact us. A single order can contain items from different sellers, and these items may arrive in separate packages on different days.

3. Payment Methods
We accept UPI, debit cards, credit cards, net banking, ShopEasy Wallet and Cash on Delivery. EMI is available on credit cards for orders above Rs 3,000, with tenures of 3, 6, 9 and 12 months. No-cost EMI is offered only on selected products and is shown on the product page. Cash on Delivery is available on orders up to Rs 10,000. For Cash on Delivery orders, please keep exact change ready or pay by UPI at the doorstep. A Cash on Delivery handling fee of Rs 20 applies to orders below Rs 499.

4. ShopEasy Wallet
ShopEasy Wallet is a prepaid balance linked to your account. You can add money to the wallet using UPI, cards or net banking. Refunds credited to the wallet are added instantly. Wallet balance never expires and can be used on any order. Wallet balance cannot be transferred to a bank account, except for refunds of Cash on Delivery orders as described in Section 11.

5. Delivery and Shipping
Delivery is free on orders above Rs 499. Orders below Rs 499 have a delivery fee of Rs 49. Standard delivery takes 3 to 5 working days. Express delivery takes 1 to 2 working days and costs Rs 99. Express delivery is available only in metro cities: Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune. Delivery to remote areas, including parts of the North East, Ladakh and the Andaman and Nicobar Islands, may take up to 10 working days. Large items such as furniture and appliances are delivered within 7 to 12 working days and include free installation where mentioned on the product page.

6. Tracking Your Order
You can track every order in the My Orders section of the app or website. You will receive updates when your order is packed, shipped, out for delivery and delivered. If an order shows as delivered but you have not received it, please report it within 48 hours of the delivery update. We will investigate with our delivery partner and respond within 3 working days.

7. Order Cancellation
You can cancel an order free of charge any time before it is shipped. Once an order is shipped, it cannot be cancelled, but you can refuse the delivery or request a return after delivery, as per the returns policy. For prepaid orders, the cancellation refund follows the same timelines as returns in Section 11. Made-to-order and personalised items cannot be cancelled once production has started.

8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery. All other clothing and footwear can be returned within 15 days of delivery. Home and kitchen products can be returned within 10 days of delivery if they are unused and in their original packaging. Innerwear, swimwear, cosmetics, perfumes and personalised items cannot be returned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip.

9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead.

10. Quality Check on Returns
Every returned item goes through a quality check at our warehouse. The check usually takes 1 to 2 working days after the item reaches us. If the item is used, damaged by the customer, or missing parts, tags or accessories, the return may be rejected and the item will be shipped back to you. You will be informed of the result by SMS and email.

11. Refunds Policy
Refunds are processed after the returned item passes the quality check. For prepaid orders, the money goes back to the original payment method within 5 to 7 working days. For UPI payments, refunds usually reach your account within 2 to 3 working days. For Cash on Delivery orders, the refund is sent to your bank account within 7 to 10 working days. You will need to add your bank account details in the app for this. You can also choose ShopEasy Wallet credit, which is added instantly. Delivery fees are refunded only if the entire order is returned because of a mistake on our side, such as a wrong or damaged item.

12. Damaged, Defective or Wrong Items
If you receive a damaged, defective or wrong item, please report it within 48 hours of delivery with clear photos of the item and the package. We will arrange a free pickup and send a replacement, or issue a full refund if a replacement is not available. Do not throw away the original packaging until the issue is resolved.

13. Electronics Policy
Electronics cannot be returned for a refund. Damaged or defective electronics are replaced within 10 days of delivery after a technician visit or online verification. All electronics carry a 1 year manufacturer warranty unless stated otherwise on the product page. For warranty claims after 10 days, contact the brand service centre listed on the product page. ShopEasy Care, our extended warranty plan, adds one more year of protection and can be bought within 30 days of delivery.

14. ShopEasy Plus Membership
ShopEasy Plus is our loyalty membership. It costs Rs 499 per year. Plus members get free express delivery in metro cities, early access to sales 24 hours before everyone else, and 5 percent extra cashback on all orders paid by UPI, credited to ShopEasy Wallet. Plus members also get priority customer care. The membership can be cancelled within 14 days of purchase for a full refund if no benefits have been used.

15. Coupons and Offers
Only one coupon can be applied per order. Coupons cannot be combined with other coupons, but they can be used along with bank offers unless stated otherwise. Coupon discounts are adjusted proportionally if part of an order is returned. Cashback is credited to ShopEasy Wallet within 7 days after the return window closes.

16. Festive Sales
During festive sales such as the Diwali Sale and the Black Friday Sale, delivery timelines may be 2 to 3 working days longer than usual. The return windows in Section 8 remain the same during sales. Sale prices are valid only while stocks last and only during the sale period.

17. Gift Cards
ShopEasy Gift Cards are available in values from Rs 500 to Rs 10,000. A gift card is valid for 12 months from the date of purchase. Gift cards cannot be redeemed for cash and cannot be used to buy other gift cards. If an order paid by gift card is returned, the amount is credited back to ShopEasy Wallet.

18. Account and Privacy
We use your personal information only to process orders, provide customer support and improve our services. We never sell your personal data. You can download your data or request deletion of your account from the Privacy section in Settings. Account deletion is completed within 30 days, after any open orders, returns or refunds are closed.

19. Marketplace Sellers
Many products on ShopEasy are sold by independent sellers. The seller name is shown on every product page. All sellers must follow this handbook for returns, refunds and warranties. ShopEasy Assured products are stored in our own warehouses, checked for quality before dispatch and usually delivered faster. If you have a problem with a seller, contact ShopEasy customer care and not the seller directly, so that we can track and resolve the issue.

20. Corporate and Bulk Orders
Businesses can place bulk orders of 50 units or more through the ShopEasy for Business portal. Bulk orders get a GST invoice with the company GSTIN, a dedicated account manager and special pricing on request. Bulk orders cannot be paid by Cash on Delivery. Returns on bulk orders are accepted only for damaged or defective items reported within 7 days of delivery.

21. Installation and Services
Free installation is provided for air conditioners, washing machines, televisions and water purifiers where mentioned on the product page. Installation is scheduled within 48 hours of delivery in metro cities and within 5 working days elsewhere. Installation does not include extra materials such as copper pipes, wall stands or extension cables, which are charged separately by the technician as per the rate card shared before work begins.

22. Delivery Instructions and Address Changes
You can add delivery instructions, such as a landmark or a preferred time, when placing an order. The delivery address can be changed only before the order is shipped. After shipping, the address cannot be changed, but you can ask the delivery partner to hold the package at a nearby ShopEasy pickup point for up to 5 days. If the delivery fails three times because the customer is unavailable, the order is returned to us and a refund is processed as per Section 11.

23. Product Reviews
Only customers who have bought and received a product can post a review. Reviews are checked for abusive language and fake content before they are published. Sellers cannot edit or remove reviews. You can edit or delete your own review at any time from the My Reviews section.

24. Frequently Asked Questions
Can I return a dress after 7 days? No. Dresses, suits and party wear can only be returned within 7 days of delivery.
Can I change the payment method after placing an order? No. Please cancel the order before it is shipped and place a new one.
Do I pay a fee for returns? No. Returns and pickups are free on all eligible items.
Can I get an invoice for my order? Yes. A downloadable GST invoice is available in My Orders after delivery.
What if my refund is delayed? If you have not received your refund within the timelines in Section 11, contact customer care with your order ID.

25. Customer Care
Customer care: 1800-000-4321 (toll free), 9 AM to 9 PM, all days.
Email: support@shopeasy.example
For order issues, chat support in the ShopEasy app is the fastest option, with an average response time of under 2 minutes. Plus members are routed to a priority queue. Complaints that are not resolved within 7 working days can be escalated to our Grievance Officer at grievance@shopeasy.example.

---
## 3 · The simplest fix: put the document in the prompt

### The whole handbook

In [40]:

handbook = open("shopeasy_handbook.txt").read()
question = "I bought a dress 12 days ago. Can I still return it?"

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{handbook}
</context>"""),
    ("user", question),
])

print(response.content)
print("Input tokens:", response.usage_metadata["input_tokens"])

No. Dresses, suits and party wear can only be returned within 7 days of delivery. Since it’s been 12 days, the return window has passed.
Input tokens: 2465


### Only the relevant section

In [39]:
question = "I bought a dress 12 days ago. Can I still return it?"

returns_policy = """8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery.
All other clothing and footwear can be returned within 15 days of delivery.
Home and kitchen products can be returned within 10 days of delivery if they are unused and in their original packaging.
Innerwear, swimwear, cosmetics, perfumes and personalised items cannot be returned for hygiene reasons.
Items must be unused and in the same condition in which they were delivered.
Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox.
To start a return, go to My Orders, select the item and tap Return.
A pickup is scheduled within 2 working days.
Please keep the item packed and ready with the order ID written on a slip."""

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{returns_policy}
</context>"""),
    ("user", question),
])

print(response.content)
print("Input tokens:", response.usage_metadata["input_tokens"])

I’m sorry, but dresses can only be returned within 7 days of delivery. Since it’s been 12 days, the return window has passed.
Input tokens: 283


In [38]:
returns_policy = """8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery.
All other clothing and footwear can be returned within 15 days of delivery.
Home and kitchen products can be returned within 10 days of delivery if they are unused and in their original packaging.
Innerwear, swimwear, cosmetics, perfumes and personalised items cannot be returned for hygiene reasons.
Items must be unused and in the same condition in which they were delivered.
Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox.
To start a return, go to My Orders, select the item and tap Return.
A pickup is scheduled within 2 working days.
Please keep the item packed and ready with the order ID written on a slip."""

response = llm.invoke([
    ("system", """You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know."""),

    ("user", f"""<context>{returns_policy}</context> Question: {question}"""),
])

print(response.content)
print("Input tokens:", response.usage_metadata["input_tokens"])

I’m sorry, but you’re outside the allowed return window. Footwear can only be returned (or exchanged) within 15 days of delivery, and your shoes were delivered 20 days ago. Therefore we can’t process an exchange at this time.
Input tokens: 294


### The answer can sit in more than one section

In [37]:
size_exchange_policy = """9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window.
The new size is delivered when the original item is picked up.
Size exchange is subject to stock availability.
If the requested size is out of stock, a refund is processed instead."""

question = "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{size_exchange_policy}
</context>"""),
    ("user", question),
])

print(response.content)

Yes—you can request one free size exchange for footwear as long as you’re still inside the return window. If you’re within that period, the new size will be shipped to you when the original pair is picked up. The exchange depends on the larger size being in stock; if it isn’t, a refund will be issued instead.


In [36]:
returns_policy = """8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery.
All other clothing and footwear can be returned within 15 days of delivery.
Home and kitchen products can be returned within 10 days of delivery if they are unused and in their original packaging.
Innerwear, swimwear, cosmetics, perfumes and personalised items cannot be returned for hygiene reasons.
Items must be unused and in the same condition in which they were delivered.
Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox.
To start a return, go to My Orders, select the item and tap Return.
A pickup is scheduled within 2 working days.
Please keep the item packed and ready with the order ID written on a slip."""

size_exchange_policy = """9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window.
The new size is delivered when the original item is picked up.
Size exchange is subject to stock availability.
If the requested size is out of stock, a refund is processed instead."""

In [ ]:
question = "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"

context = size_exchange_policy + "\n\n" + returns_policy      # returns_policy is defined in the cell above

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
    ("user", question),
])

print(response.content)

I’m sorry, but the free size‑exchange for footwear is only available within the return window. Since shoes can be returned (or exchanged) within 15 days of delivery and yours were received 20 days ago, the exchange period has passed. Therefore a size exchange isn’t possible at this time.


> **Takeaway:** Sending only the relevant pieces gives the same answer at a fraction of the tokens. But here *we* picked the sections. For any question, the computer has to find them: that's retrieval.

### RAG has two pipelines
**Indexing** (offline, done once, re-run when documents change): load → chunk → embed → store

**Answering** (on every question): embed the question → retrieve the top-k chunks → add them to the prompt → LLM answers

The next four sections build the indexing pipeline. Section 8 puts the two together.

---
## 4 · Chunking: split the document into pieces

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Character-based splitting is the simplest approach to text splitting.

splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)
chunks = splitter.create_documents([handbook])

print(len(chunks), "chunks")
for chunk in chunks[:10]:
    print(chunk.page_content, "\n-----")

25 chunks
ShopEasy Customer Policy Handbook
Version 3.2, effective 1 October 2026

1. About ShopEasy
ShopEasy is an online shopping platform for fashion, home, beauty and electronics. We deliver to more than 19,000 pin codes across India. This handbook explains how orders, payments, deliveries, returns, refunds and warranties work. If anything in this handbook is unclear, our customer care team is happy to help.

2. Placing an Order
You can place an order on the ShopEasy website or the ShopEasy app. You n 
-----
on the ShopEasy website or the ShopEasy app. You need a registered account with a verified mobile number. Once an order is placed, you will receive an order confirmation by SMS, email and app notification. Each order has a unique order ID, for example SE-4521-889. Please keep this ID handy when you contact us. A single order can contain items from different sellers, and these items may arrive in separate packages on different days.

3. Payment Methods
We accept UPI, debit cards

#### Converting the chunks to embedding

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# InMemoryVectorStore, a lightweight vector store that keeps all embeddings and documents in the system memory (RAM).

vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

In [ ]:
chunk_id, stored = list(vector_store.store.items())[0]

print("TEXT:  ", stored["text"][:])
print("VECTOR:", stored["vector"][:5], "...", len(stored["vector"]), "numbers")

TEXT:   ShopEasy Customer Policy Handbook
Version 3.2, effective 1 October 2026

1. About ShopEasy
ShopEasy is an online shopping platform for fashion, home, beauty and electronics. We deliver to more than 19,000 pin codes across India. This handbook explains how orders, payments, deliveries, returns, refunds and warranties work. If anything in this handbook is unclear, our customer care team is happy to help.

2. Placing an Order
You can place an order on the ShopEasy website or the ShopEasy app. You n
VECTOR: [-0.03542464226484299, -0.050875913351774216, -0.02688375487923622, -0.004126184619963169, -0.041744034737348557] ... 384 numbers


In [ ]:
import pandas as pd

df = pd.DataFrame(vector_store.store.values())
print(df.shape)
df

(25, 4)


,id,vector,text,metadata
0,3a240b81-f9e6-4cd7-b4a6-c4595059f29a,"[-0.03542464226484299, -0.050875913351774216, ...",ShopEasy Customer Policy Handbook\nVersion 3.2...,{}
1,f7aee62d-2fda-4deb-ba65-263921013ef6,"[-0.07791482657194138, -0.002146280836313963, ...",on the ShopEasy website or the ShopEasy app. Y...,{}
2,53e811ad-0a5a-48d2-b0ea-4d8d1df360aa,"[0.00302444351837039, -0.02560870535671711, -0...","thods\nWe accept UPI, debit cards, credit card...",{}
3,972d0995-0b2f-48a8-a946-e9b652f745ad,"[-0.034105200320482254, 0.001109780976548791, ...",ndling fee of Rs 20 applies to orders below Rs...,{}
4,3c64e05a-179f-4587-9cfb-af6f489ce810,"[0.04964378476142883, -0.03905325010418892, 0....",y and Shipping\nDelivery is free on orders abo...,{}
5,173ce0b2-2c6c-48df-9645-d0bd52e38c57,"[0.0060388436540961266, -0.03870180994272232, ...",10 working days. Large items such as furniture...,{}
6,b7cec973-f162-4669-9d30-9895c94251fb,"[-0.04047859460115433, 0.04868895560503006, 0....",hours of the delivery update. We will investig...,{}
7,476e7478-1320-4b56-81ab-3963702c9760,"[-0.06174840033054352, 0.05229303985834122, 0....",e-to-order and personalised items cannot be ca...,{}
8,334075b9-a746-472f-9794-2a8f72ffecec,"[-0.08355501294136047, 0.0363311804831028, 0.0...",rned for hygiene reasons. Items must be unused...,{}
9,2e2a12ea-2ced-408f-9d81-8e989dbbc746,"[-0.002238855231553316, -0.011502721346914768,...","ear, you can request one free size exchange wi...",{}


In [ ]:
results = vector_store.similarity_search_with_score("I want my money back for a dress", k=3)

for doc, score in results:
    print(f"----- score {score:.2f}")
    print(doc.page_content)

----- score 0.48
rned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip.

9. Size Exchange
For clothing and footwear, you can request one free size exchange within
----- score 0.46
post a review. Reviews are checked for abusive language and fake content before they are published. Sellers cannot edit or remove reviews. You can edit or delete your own review at any time from the My Reviews section.

24. Frequently Asked Questions
Can I return a dress after 7 days? No. Dresses, suits and party wear can only be returned within 7 days of delivery.
Can I change the payment method after placing an order? No. Please cancel the order before it is 

In [ ]:
results = vector_store.similarity_search_with_score("Is express delivery available in Jaipur?", k=3)

for doc, score in results:
    print(f"----- score {score:.2f}")
    print(doc.page_content)

----- score 0.57
y and Shipping
Delivery is free on orders above Rs 499. Orders below Rs 499 have a delivery fee of Rs 49. Standard delivery takes 3 to 5 working days. Express delivery takes 1 to 2 working days and costs Rs 99. Express delivery is available only in metro cities: Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune. Delivery to remote areas, including parts of the North East, Ladakh and the Andaman and Nicobar Islands, may take up to 10 working days. Large items such as furniture an
----- score 0.53
free express delivery in metro cities, early access to sales 24 hours before everyone else, and 5 percent extra cashback on all orders paid by UPI, credited to ShopEasy Wallet. Plus members also get priority customer care. The membership can be cancelled within 14 days of purchase for a full refund if no benefits have been used.

15. Coupons and Offers
Only one coupon can be applied per order. Coupons cannot be combined with other coupons, but they can be used 

In [ ]:
question = "I want to cancel my order before it ships -- will I get my refund right away?"

print("k=1:")
for doc, score in vector_store.similarity_search_with_score(question, k=1):
    print(f"[score {score:.2f}]")
    print(doc.page_content, "\n-----")

print("\nk=3:")
for doc, score in vector_store.similarity_search_with_score(question, k=3):
    print(f"[score {score:.2f}]")
    print(doc.page_content, "\n-----")

k=1:
[score 0.70]
hours of the delivery update. We will investigate with our delivery partner and respond within 3 working days.

7. Order Cancellation
You can cancel an order free of charge any time before it is shipped. Once an order is shipped, it cannot be cancelled, but you can refuse the delivery or request a return after delivery, as per the returns policy. For prepaid orders, the cancellation refund follows the same timelines as returns in Section 11. Made-to-order and personalised items cannot be cancel 
-----

k=3:
[score 0.70]
hours of the delivery update. We will investigate with our delivery partner and respond within 3 working days.

7. Order Cancellation
You can cancel an order free of charge any time before it is shipped. Once an order is shipped, it cannot be cancelled, but you can refuse the delivery or request a return after delivery, as per the returns policy. For prepaid orders, the cancellation refund follows the same timelines as returns in Section 11. Made-to-or

In The above example at K = 1 we are retrieving the chunk which points to section 11 without actual answer

But at K = 3 it pulls the actual answer with number

In [ ]:
question = "Is express delivery available in Jaipur?"

results = vector_store.similarity_search(question, k=3)
context = "\n\n".join(doc.page_content for doc in results)

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
    ("user", question),
])

print(response.content)
print("Input tokens:", response.usage_metadata["input_tokens"])

Express delivery is not available in Jaipur. It’s only offered in the metro cities listed in the policy (Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune).
Input tokens: 462


In [ ]:
question = "What is ShopEasy's office address in Mumbai?"

results = vector_store.similarity_search(question, k=3)
context = "\n\n".join(doc.page_content for doc in results)

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
    ("user", question),
])

print(response.content)
print("Input tokens:", response.usage_metadata["input_tokens"])

I don't know.
Input tokens: 452


In [ ]:
question = "I want to cancel my order before it ships -- will I get my refund right away?"

for k in [1, 3]:
    results = vector_store.similarity_search(question, k=k)
    context = "\n\n".join(doc.page_content for doc in results)

    response = llm.invoke([
        ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
        ("user", question),
    ])

    print(f"===== k={k} =====")
    print(response.content, "\n")

===== k=1 =====
You can cancel your order free of charge as long as it hasn’t been shipped yet. The refund for a prepaid order isn’t issued immediately – it follows the same timeline as a standard return (as set out in Section 11 of our policy). If the order has already shipped, it can’t be cancelled, but you can refuse the delivery or arrange a return after it arrives. 

===== k=3 =====
You can cancel your order free of charge any time **before it is shipped**.  
The refund won’t be instant – it follows the same schedule as a normal return:

* **Pre‑paid orders (card, net‑banking, etc.)** – the amount is returned to the original payment method in **5‑7 working days**.  
* **UPI payments** – the refund usually reaches your account in **2‑3 working days**.  
* **Cash on Delivery** – the refund is sent to your bank account in **7‑10 working days** (or you can opt for an instant credit to your ShopEasy Wallet).

So, after you cancel, expect the refund according to the timeline above rathe

In the above example we can see that retrieving only one chunk (K = 1) gives us correct information but its incomplete
But when retrieving Three chunks (K = 3) We get complete details


In [ ]:
question = "Is express delivery available in Jaipur?"

results = vector_store.similarity_search_with_score(question, k=3)
context = "\n\n".join(doc.page_content for doc, score in results)

system_prompt = f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""

print("===== Full prompt sent to the LLM =====")
print("SYSTEM:", system_prompt)
print("USER:", question)

response = llm.invoke([
    ("system", system_prompt),
    ("user", question),
])

print("\n===== Answer =====")
print(response.content)
print("Input tokens:", response.usage_metadata["input_tokens"])

===== Full prompt sent to the LLM =====
SYSTEM: You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
y and Shipping
Delivery is free on orders above Rs 499. Orders below Rs 499 have a delivery fee of Rs 49. Standard delivery takes 3 to 5 working days. Express delivery takes 1 to 2 working days and costs Rs 99. Express delivery is available only in metro cities: Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune. Delivery to remote areas, including parts of the North East, Ladakh and the Andaman and Nicobar Islands, may take up to 10 working days. Large items such as furniture an

free express delivery in metro cities, early access to sales 24 hours before everyone else, and 5 percent extra cashback on all orders paid by UPI, credited to ShopEasy Wallet. Plus members also get priority customer care. The membership can be cancelled within 14 days of purchase for a full refund if no b

# 1. Retrieve

In [ ]:
question = "Is express delivery available in Jaipur?"

results = vector_store.similarity_search_with_score(question, k=3)

for doc, score in results:
    print(f"[score {score:.2f}]")
    print(doc.page_content, "\n-----")

[score 0.57]
y and Shipping
Delivery is free on orders above Rs 499. Orders below Rs 499 have a delivery fee of Rs 49. Standard delivery takes 3 to 5 working days. Express delivery takes 1 to 2 working days and costs Rs 99. Express delivery is available only in metro cities: Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune. Delivery to remote areas, including parts of the North East, Ladakh and the Andaman and Nicobar Islands, may take up to 10 working days. Large items such as furniture an 
-----
[score 0.53]
free express delivery in metro cities, early access to sales 24 hours before everyone else, and 5 percent extra cashback on all orders paid by UPI, credited to ShopEasy Wallet. Plus members also get priority customer care. The membership can be cancelled within 14 days of purchase for a full refund if no benefits have been used.

15. Coupons and Offers
Only one coupon can be applied per order. Coupons cannot be combined with other coupons, but they can be used a

# 2. Augment

In [ ]:
context = "\n\n".join(doc.page_content for doc, score in results)

system_prompt = f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""

print("SYSTEM:", system_prompt)
print("\nUSER:", question)

SYSTEM: You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
y and Shipping
Delivery is free on orders above Rs 499. Orders below Rs 499 have a delivery fee of Rs 49. Standard delivery takes 3 to 5 working days. Express delivery takes 1 to 2 working days and costs Rs 99. Express delivery is available only in metro cities: Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune. Delivery to remote areas, including parts of the North East, Ladakh and the Andaman and Nicobar Islands, may take up to 10 working days. Large items such as furniture an

free express delivery in metro cities, early access to sales 24 hours before everyone else, and 5 percent extra cashback on all orders paid by UPI, credited to ShopEasy Wallet. Plus members also get priority customer care. The membership can be cancelled within 14 days of purchase for a full refund if no benefits have been used.

15. Coupons and

# 3. Generate



In [ ]:
response = llm.invoke([
    ("system", system_prompt),
    ("user", question),
])

print(response.content)
print("\nInput tokens:", response.usage_metadata["input_tokens"])

Express delivery is not available in Jaipur. It’s only offered in the metro cities listed (Delhi NCR, Mumbai, Bengaluru, Chennai, Hyderabad, Kolkata and Pune).

Input tokens: 462


---
## Observability: see every step in LangSmith
Open LangSmith → project **shopeasy-rag**. For each question you'll see:

- **ChatGroq:** the full prompt (context + question), the answer, tokens and latency

> When a RAG answer is wrong, first check: **were the right chunks retrieved?**

---
## When retrieval goes wrong

In [ ]:
question = "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"

results = vector_store.similarity_search_with_score(question, k=3)

for doc, score in results:
    print(f"[score {score:.2f}]")
    print(doc.page_content, "\n-----")

[score 0.45]
rned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip.

9. Size Exchange
For clothing and footwear, you can request one free size exchange within 
-----
[score 0.37]
ear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead.

10. Quality Check on Returns
Every returned item goes through a quality check at our warehouse. The check usually takes 1 to 2 working days after the item reaches us. If the item is used, damaged by the customer, o

In [ ]:
context = "\n\n".join(doc.page_content for doc, score in results)

system_prompt = f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""

print("SYSTEM:", system_prompt)
print("\nUSER:", question)

SYSTEM: You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
rned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip.

9. Size Exchange
For clothing and footwear, you can request one free size exchange within

ear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead.

10. Quality Check on Returns
Every returned item goes through a quality check at our warehouse.

In [ ]:
response = llm.invoke([
    ("system", system_prompt),
    ("user", question),
])

print(response.content)

Yes—you can request a free size exchange for your running shoes as long as you’re still within the return window.  

**How to proceed**

1. Open the **My Orders** section in the ShopEasy app/website.  
2. Find the shoes you received and tap **Return**.  
3. Choose the **size‑exchange** option and select the larger size you need.  
4. A pickup will be scheduled within 2 working days. Keep the shoes packed, include the order‑ID slip, and have them ready for the courier.  

The new size will be delivered when the original pair is picked up. The exchange is subject to stock availability; if the requested size is out of stock, a refund will be processed instead.


> **Takeaway:** Same LLM, same question. Retrieval quality decides answer quality. Next: better chunking.

### Strategy 1 · Recursive splitting
Instead of cutting every 500 characters, `RecursiveCharacterTextSplitter` tries to cut at a blank line first, then a line break, then a space. Words stay whole.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
recursive_chunks = splitter.create_documents([handbook])
recursive_store = InMemoryVectorStore.from_documents(recursive_chunks, embeddings)

print(len(recursive_chunks), "chunks\n")
for chunk in recursive_chunks[0:5]:
    print(chunk.page_content, "\n-----")

35 chunks

ShopEasy Customer Policy Handbook
Version 3.2, effective 1 October 2026

1. About ShopEasy
ShopEasy is an online shopping platform for fashion, home, beauty and electronics. We deliver to more than 19,000 pin codes across India. This handbook explains how orders, payments, deliveries, returns, refunds and warranties work. If anything in this handbook is unclear, our customer care team is happy to help. 
-----
2. Placing an Order
You can place an order on the ShopEasy website or the ShopEasy app. You need a registered account with a verified mobile number. Once an order is placed, you will receive an order confirmation by SMS, email and app notification. Each order has a unique order ID, for example SE-4521-889. Please keep this ID handy when you contact us. A single order can contain items from different sellers, and these items may arrive in separate packages on different days. 
-----
3. Payment Methods 
-----
We accept UPI, debit cards, credit cards, net banking, ShopEasy 

In [ ]:
question = "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"

for doc, score in recursive_store.similarity_search_with_score(question, k=3):
    print(f"[score {score:.2f}]")
    print(doc.page_content, "\n-----")

[score 0.52]
9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead. 
-----
[score 0.35]
Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip. 
-----
[score 0.25]
24. Frequently Asked Questions
Can I return a dress after 7 days? No. Dresses, suits and party wear can only be returned within 7 days of delivery.
Can I change the payment method after placing an order? No. Please cancel the order before it is shipped and place a new one.
Do I pay a fee for returns? No. Returns and pickups are free on all eligible

In [ ]:
context = "\n\n".join(doc.page_content for doc in recursive_store.similarity_search(question, k=3))

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
    ("user", question),
])

print(response.content)

You can request a free size exchange for footwear as long as you’re still within the return window. Since you received the shoes 20 days ago, you’re still eligible to start a size‑exchange request.

**What to do:**

1. Open the **My Orders** section in the app/website.  
2. Select the running‑shoes item and tap **Return**.  
3. Choose the size‑exchange option and specify the new size you need.  
4. A pickup will be scheduled within 2 working days. Keep the shoes packed, include the order‑ID slip, and make sure they’re in the original shoebox.  
5. When the original pair is collected, the new size will be delivered (subject to stock availability). If the requested size is out of stock, a refund will be processed instead.  

If you have any trouble, let us know!


> **Takeaway:** The "footwear… 15 days" rule is in chunk #4, just outside our top 3. Splitting cut it away from its heading, so it starts with "Dresses, suits and party wear…" and looks like a dresses rule to the embedding model. We could raise k, but that sends more text on every question. The better fix is chunks that keep each rule together with its heading.

In [ ]:
# K is set to 4
question = "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"

for doc, score in recursive_store.similarity_search_with_score(question, k=4):
    print(f"[score {score:.2f}]")
    print(doc.page_content, "\n-----")

[score 0.52]
9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead. 
-----
[score 0.35]
Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip. 
-----
[score 0.25]
24. Frequently Asked Questions
Can I return a dress after 7 days? No. Dresses, suits and party wear can only be returned within 7 days of delivery.
Can I change the payment method after placing an order? No. Please cancel the order before it is shipped and place a new one.
Do I pay a fee for returns? No. Returns and pickups are free on all eligible

In [ ]:
context = "\n\n".join(doc.page_content for doc in recursive_store.similarity_search(question, k=4))

response = llm.invoke([
    ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
    ("user", question),
])

print(response.content)

I’m sorry, but the free size‑exchange for footwear is only available within the 15‑day return window. Since you received the shoes 20 days ago, the exchange period has passed, so a size exchange isn’t possible.


### Strategy 2 · Recursive splitting with structure - Contextual headers
Small chunks are precise, but they lose context ("It costs Rs 499 per year": what costs?). Fix: keep chunks small, and stamp every chunk with where it came from.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)

contextual_chunks = []
for section in handbook.split("\n\n")[1:]:
    heading = section.split("\n")[0]
    for piece in splitter.split_text(section):
        if piece.strip() == heading:          # skip chunks that are only the heading
            continue
        contextual_chunks.append(Document(
            page_content=f"ShopEasy Handbook > {heading}\n{piece}",
            metadata={"section": heading},
        ))

contextual_store = InMemoryVectorStore.from_documents(contextual_chunks, embeddings)

print(len(contextual_chunks), "chunks\n")

for chunk in contextual_chunks[18:23]:
    print(chunk.page_content, "\n-----")

51 chunks

ShopEasy Handbook > 9. Size Exchange
9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead. 
-----
ShopEasy Handbook > 10. Quality Check on Returns
Every returned item goes through a quality check at our warehouse. The check usually takes 1 to 2 working days after the item reaches us. If the item is used, damaged by the customer, or missing parts, tags or accessories, the return may be rejected and the item will be shipped back to you. You 
-----
ShopEasy Handbook > 10. Quality Check on Returns
will be informed of the result by SMS and email. 
-----
ShopEasy Handbook > 11. Refunds Policy
Refunds are processed after the returned item passes the quality check. For prepaid orders, the money goes back to the original payment method within 5 to 7 wo

In [ ]:
for question in [
    "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?",
    "How much does ShopEasy Plus membership cost?",
]:
    print(f"===== {question} =====")

    results = contextual_store.similarity_search_with_score(question, k=3)
    for doc, score in results:
        print(f"[score {score:.2f}]")
        print(doc.page_content, "\n-----")

    context = "\n\n".join(doc.page_content for doc, score in results)
    response = llm.invoke([
        ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
        ("user", question),
    ])
    print("A:", response.content, "\n")

===== My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size? =====
[score 0.54]
ShopEasy Handbook > 9. Size Exchange
9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead. 
-----
[score 0.32]
ShopEasy Handbook > 8. Returns Policy
cosmetics, perfumes and personalised items cannot be returned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, 
-----
[score 0.29]
ShopEasy Handbook > 8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery. All other clothing and footwear can be returned within 15 days

> **Takeaway:** Same small chunks, plus one line saying where each came from. "It costs Rs 499 per year" now carries "ShopEasy Plus Membership" with it, so it matches the question. Both questions answered correctly. This is cheap, simple and widely used in production.

### Strategy 3 · Section-aware chunking
Instead of cutting by character count at all, split along the document's own structure: each numbered section becomes one chunk.

In [ ]:
section_chunks = [
    Document(page_content=section, metadata={"section": section.split("\n")[0]})
    for section in handbook.split("\n\n")[1:]
]
section_store = InMemoryVectorStore.from_documents(section_chunks, embeddings)

print(len(section_chunks), "chunks\n")

for chunk in section_chunks:
    print(chunk.page_content, "\n-----")

25 chunks

1. About ShopEasy
ShopEasy is an online shopping platform for fashion, home, beauty and electronics. We deliver to more than 19,000 pin codes across India. This handbook explains how orders, payments, deliveries, returns, refunds and warranties work. If anything in this handbook is unclear, our customer care team is happy to help. 
-----
2. Placing an Order
You can place an order on the ShopEasy website or the ShopEasy app. You need a registered account with a verified mobile number. Once an order is placed, you will receive an order confirmation by SMS, email and app notification. Each order has a unique order ID, for example SE-4521-889. Please keep this ID handy when you contact us. A single order can contain items from different sellers, and these items may arrive in separate packages on different days. 
-----
3. Payment Methods
We accept UPI, debit cards, credit cards, net banking, ShopEasy Wallet and Cash on Delivery. EMI is available on credit cards for orders above R

In [ ]:
import pandas as pd

df = pd.DataFrame(section_store.store.values())
print(df.shape)
df

(25, 4)


,id,vector,text,metadata
0,c4b885fa-a2d3-42f5-9032-1d9be67421ce,"[-0.026119165122509003, -0.05842222645878792, ...",1. About ShopEasy\nShopEasy is an online shopp...,{'section': '1. About ShopEasy'}
1,fc8b7108-0457-4ceb-a4d8-aaf40169957d,"[-0.07707410305738449, -0.02945384569466114, -...",2. Placing an Order\nYou can place an order on...,{'section': '2. Placing an Order'}
2,8532ea4c-ce97-4bfa-8f21-354aef1195d3,"[0.0303182452917099, -0.03579355403780937, -0....","3. Payment Methods\nWe accept UPI, debit cards...",{'section': '3. Payment Methods'}
3,a34773c7-a453-4ed5-9977-c63439eef7c1,"[-0.07548075914382935, 0.01307669933885336, -0...",4. ShopEasy Wallet\nShopEasy Wallet is a prepa...,{'section': '4. ShopEasy Wallet'}
4,8048e648-bed0-4155-b10a-512eb546a727,"[0.04510926827788353, -0.015346833504736423, 0...",5. Delivery and Shipping\nDelivery is free on ...,{'section': '5. Delivery and Shipping'}
5,18bb4196-0f8b-4c44-9e8c-3d5f1ddb1d9f,"[-0.03627987578511238, -0.059057194739580154, ...",6. Tracking Your Order\nYou can track every or...,{'section': '6. Tracking Your Order'}
6,21f108e6-461d-4fe0-8870-7dd887eed812,"[-0.03764280304312706, 0.055805668234825134, 0...",7. Order Cancellation\nYou can cancel an order...,{'section': '7. Order Cancellation'}
7,76d7d419-3f7c-44bf-8e50-adb3e9ff6460,"[-0.07717323303222656, 0.034063227474689484, 0...","8. Returns Policy\nDresses, suits and party we...",{'section': '8. Returns Policy'}
8,8e9eb193-a8d6-4ca4-9248-20ddf77d27ff,"[-0.046713948249816895, -0.011522669345140457,...","9. Size Exchange\nFor clothing and footwear, y...",{'section': '9. Size Exchange'}
9,83e6e648-1877-4b81-99f9-7756921edddc,"[-0.007796527352184057, -0.010378343984484673,...",10. Quality Check on Returns\nEvery returned i...,{'section': '10. Quality Check on Returns'}


In [ ]:
print(section_chunks[7].page_content)      # 8. Returns Policy, complete

8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery. All other clothing and footwear can be returned within 15 days of delivery. Home and kitchen products can be returned within 10 days of delivery if they are unused and in their original packaging. Innerwear, swimwear, cosmetics, perfumes and personalised items cannot be returned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item and tap Return. A pickup is scheduled within 2 working days. Please keep the item packed and ready with the order ID written on a slip.


In [ ]:
question = "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"

results = section_store.similarity_search_with_score(question, k=2)

for doc, score in results:
    print(f"[score {score:.2f}] {doc.metadata['section'] }\n")
    print(doc.page_content, "\n-----")

[score 0.52] 9. Size Exchange

9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead. 
-----
[score 0.28] 8. Returns Policy

8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery. All other clothing and footwear can be returned within 15 days of delivery. Home and kitchen products can be returned within 10 days of delivery if they are unused and in their original packaging. Innerwear, swimwear, cosmetics, perfumes and personalised items cannot be returned for hygiene reasons. Items must be unused and in the same condition in which they were delivered. Dresses and suits must be returned with all original tags attached. Footwear must be returned in the original shoebox. To start a return, go to My Orders, select the item 

In [ ]:
context = "\n\n".join(doc.page_content for doc, score in results)

system_prompt = f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""

response = llm.invoke([
    ("system", system_prompt),
    ("user", question),
])

print(response.content)
print("\nSources:", [doc.metadata["section"] for doc, score in results])

I’m sorry, but the free size‑exchange option is only available within the return window. For footwear, the return window is 15 days from delivery, and your shoes were received 20 days ago. Since that period has passed, a size exchange isn’t possible at this time. If you have any other questions, feel free to let us know.

Sources: ['9. Size Exchange', '8. Returns Policy']


> **Takeaway:** Same question, same model, same k or even lower k. Only the chunking changed, and the answer went from a wrong "Yes" to the correct "No". Each section is a complete rule, and the metadata gives us a source to show the customer. When a document has structure, chunk along it.

### Strategy 4 · Semantic chunking
Split where the **meaning changes**, not every N characters: embed every sentence, compare each one with the next, and start a new chunk where similarity drops.


A support call transcript has no headings, but the topic changes several times. Let's see if semantic chunking can find the topic boundaries by itself.

In [ ]:
%%writefile call_transcript.txt
Agent: Thank you for calling ShopEasy, my name is Neha. How can I help you today?
Customer: Hi Neha. My order SE-4521-889 was supposed to arrive on Monday, but it still hasn't come.
Agent: I'm sorry about that. Let me check the tracking. The parcel is at our Pune hub and was delayed by heavy rain.
Customer: When will it reach me then?
Agent: The courier now expects to deliver it by Thursday evening. You'll get an SMS when it's out for delivery.
Customer: Okay, thanks. Another thing, I have about 800 rupees in my ShopEasy Wallet. Does that expire?
Agent: No, wallet balance never expires. You can use it on any order, including sale items.
Customer: Can I move it to my bank account?
Agent: Wallet balance can't be transferred to a bank account, except refunds for Cash on Delivery orders.
Customer: Got it. Also, I changed my phone number last week. How do I update it on my account?
Agent: Go to Settings, then Profile, then Edit Phone Number. We'll send an OTP to the new number to verify it.
Customer: And will my order updates go to the new number after that?
Agent: Yes, all SMS updates will go to the verified new number from then on.
Customer: Perfect. Last question, I'm thinking of buying ShopEasy Plus. What do I get?
Agent: Plus costs 499 rupees a year. You get free express delivery in metro cities, early access to sales and 5 percent extra cashback on UPI payments.
Customer: Sounds good, I'll sign up today. Thanks for your help.
Agent: You're welcome! Have a great day.

Writing call_transcript.txt


In [ ]:
import numpy as np

transcript = open("call_transcript.txt").read()
lines = [line for line in transcript.split("\n") if line.strip()]
vectors = embeddings.embed_documents(lines)

similarity = [np.dot(vectors[i], vectors[i + 1]) for i in range(len(lines) - 1)]
threshold = np.percentile(similarity, 25)        # the lowest 25% = topic changes

chunks, current = [], [lines[0]]
for i, sim in enumerate(similarity):
    if sim < threshold:                           # topic changed: close the chunk
        chunks.append("\n".join(current))
        current = []
    current.append(lines[i + 1])
chunks.append("\n".join(current))

print(len(chunks), "chunks\n")
for chunk in chunks:
    print(chunk, "\n-----")

5 chunks

Agent: Thank you for calling ShopEasy, my name is Neha. How can I help you today?
Customer: Hi Neha. My order SE-4521-889 was supposed to arrive on Monday, but it still hasn't come.
Agent: I'm sorry about that. Let me check the tracking. The parcel is at our Pune hub and was delayed by heavy rain.
Customer: When will it reach me then?
Agent: The courier now expects to deliver it by Thursday evening. You'll get an SMS when it's out for delivery. 
-----
Customer: Okay, thanks. Another thing, I have about 800 rupees in my ShopEasy Wallet. Does that expire?
Agent: No, wallet balance never expires. You can use it on any order, including sale items.
Customer: Can I move it to my bank account?
Agent: Wallet balance can't be transferred to a bank account, except refunds for Cash on Delivery orders. 
-----
Customer: Got it. Also, I changed my phone number last week. How do I update it on my account?
Agent: Go to Settings, then Profile, then Edit Phone Number. We'll send an OTP to the 

In [ ]:
fixed_store = InMemoryVectorStore.from_texts(
    CharacterTextSplitter(separator="", chunk_size=300, chunk_overlap=0).split_text(transcript), embeddings)
semantic_store = InMemoryVectorStore.from_texts(chunks, embeddings)

question = "When will the delayed order arrive?"

for name, store in [("Fixed-size", fixed_store), ("Semantic", semantic_store)]:
    chunk = store.similarity_search(question, k=1)[0].page_content

    response = llm.invoke([
        ("system", f"""You are a ShopEasy support quality analyst.
Answer only using the call transcript excerpt below. If the answer is not there, say you don't know.

<transcript>
{chunk}
</transcript>"""),
        ("user", question),
    ])

    print(f"===== {name} =====")
    print("CHUNK:\n" + chunk)
    print("\nANSWER:", response.content, "\n")

===== Fixed-size =====
CHUNK:
Agent: Thank you for calling ShopEasy, my name is Neha. How can I help you today?
Customer: Hi Neha. My order SE-4521-889 was supposed to arrive on Monday, but it still hasn't come.
Agent: I'm sorry about that. Let me check the tracking. The parcel is at our Pune hub and was delayed by heavy rain.
C

ANSWER: I don't know. 

===== Semantic =====
CHUNK:
Agent: Thank you for calling ShopEasy, my name is Neha. How can I help you today?
Customer: Hi Neha. My order SE-4521-889 was supposed to arrive on Monday, but it still hasn't come.
Agent: I'm sorry about that. Let me check the tracking. The parcel is at our Pune hub and was delayed by heavy rain.
Customer: When will it reach me then?
Agent: The courier now expects to deliver it by Thursday evening. You'll get an SMS when it's out for delivery.

ANSWER: The courier now expects to deliver it by Thursday evening. 



> **Takeaway:** No headings, yet semantic chunking found all four topic changes on its own: delivery → wallet → phone number → Plus. Fixed-size chunks cut sentences and mixed topics. Use semantic chunking for transcripts, chats; use the structure when a document has one.

With OpenAI’s current prices (per 1M tokens: text-embedding-3-small $0.02, text-embedding-3-large $0.13)

### Strategy 5 · Parent–child (small to big)
Contracts are written in clauses. A question usually matches **one** clause, but the answer needs its neighbours: the rule, the cap, the exception. Search small (clauses), send big (the whole section).

In [ ]:
%%writefile contract.txt
MASTER SERVICES AGREEMENT between ShopEasy Pvt Ltd ("ShopEasy") and SwiftShip Logistics Pvt Ltd ("SwiftShip"), effective 1 April 2026

4. Service Levels
4.1 SwiftShip shall deliver at least 95 percent of orders to metro cities within 2 working days of pickup. 4.2 For non-metro locations, SwiftShip shall deliver at least 90 percent of orders within 5 working days of pickup. 4.3 Remote locations, including parts of the North East, Ladakh and the Andaman and Nicobar Islands, are excluded from the percentages in 4.1 and 4.2 and shall be delivered within 10 working days. 4.4 Service levels are measured per calendar month, using the delivery timestamps recorded in the ShopEasy tracking system. 4.5 SwiftShip shall share a monthly performance report with ShopEasy by the 5th working day of the following month. 4.6 During the festive peak period, defined as the Diwali week and the Black Friday week each year, every delivery time in this Section is extended by 2 working days.

5. Service Credits and Penalties
5.1 If SwiftShip misses the metro service level in 4.1 in any month, it shall credit ShopEasy 1 percent of that month's invoice for every full percentage point below 95 percent. 5.2 If SwiftShip misses the non-metro service level in 4.2, the credit is 0.5 percent of the monthly invoice for every full percentage point below 90 percent. 5.3 Total service credits in any month shall not exceed 10 percent of that month's invoice. 5.4 Service credits are adjusted against the next invoice and are not paid in cash. 5.5 If SwiftShip misses the metro service level for three consecutive months, ShopEasy may terminate this Agreement under Section 9. 5.6 Service credits are ShopEasy's only financial remedy for delays, except where the delay was caused by gross negligence.

6. Lost and Damaged Shipments
6.1 SwiftShip is responsible for every shipment from pickup until delivery is confirmed. 6.2 For shipments lost in transit, SwiftShip shall reimburse ShopEasy the declared value of the shipment, up to Rs 50,000 per shipment. 6.3 For shipments damaged in transit, SwiftShip shall reimburse the repair cost or the declared value, whichever is lower. 6.4 Claims must be raised by ShopEasy within 30 days of the expected delivery date, with photographs and the packing slip. 6.5 SwiftShip shall settle accepted claims within 15 working days. 6.6 Shipments containing jewellery, cash or documents are not covered unless declared in advance and charged at the high-value rate.

9. Termination
9.1 Either party may terminate this Agreement with 90 days' written notice. 9.2 ShopEasy may terminate immediately by written notice if SwiftShip misses the metro service level for three consecutive months, as described in 5.5. 9.3 Either party may terminate immediately if the other party becomes insolvent. 9.4 On termination, SwiftShip shall complete delivery of all shipments already picked up. 9.5 Service credits and open claims survive termination and shall be settled within 60 days.

Writing contract.txt


In [ ]:
import re
import sqlite3

contract = open("contract.txt").read()

db = sqlite3.connect("shopeasy_parents.db")
db.execute("CREATE TABLE IF NOT EXISTS parents (section TEXT PRIMARY KEY, text TEXT)")

child_chunks = []
for section in contract.split("\n\n")[1:]:
    heading, body = section.split("\n", 1)
    db.execute("INSERT OR REPLACE INTO parents VALUES (?, ?)", (heading, section))     # parent → SQLite
    for clause in re.split(r"(?=\d\.\d )", body):
        if clause.strip():
            child_chunks.append(Document(page_content=clause.strip(), metadata={"section": heading}))
db.commit()

child_store = InMemoryVectorStore.from_documents(child_chunks, embeddings)                # children → vector store

In [ ]:
import pandas as pd

pd.read_sql("SELECT section, text FROM parents", db)

,section,text
0,4. Service Levels,4. Service Levels\n4.1 SwiftShip shall deliver...
1,5. Service Credits and Penalties,5. Service Credits and Penalties\n5.1 If Swift...
2,6. Lost and Damaged Shipments,6. Lost and Damaged Shipments\n6.1 SwiftShip i...
3,9. Termination,9. Termination\n9.1 Either party may terminate...


In [ ]:
import pandas as pd

df = pd.DataFrame(child_store.store.values())
df["vector"] = df["vector"].apply(lambda v: f"{len(v)} numbers: {v[:3]} ...")
df

,id,vector,text,metadata
0,e75451b1-5377-43d2-a2f9-c6de4c93a752,"384 numbers: [0.008303877897560596, -0.0262646...",4.1 SwiftShip shall deliver at least 95 percen...,{'section': '4. Service Levels'}
1,b0633800-f5c1-4dad-87f5-92241484a730,"384 numbers: [0.007859282195568085, -0.0615895...","4.2 For non-metro locations, SwiftShip shall d...",{'section': '4. Service Levels'}
2,51bdac9e-442d-4421-94c9-915bc7669c9d,"384 numbers: [0.13478060066699982, -0.03381951...","4.3 Remote locations, including parts of the N...",{'section': '4. Service Levels'}
3,f049ccc9-6011-4835-b80d-94e4bac44c54,"384 numbers: [0.05859140679240227, -0.00763211...",4.1 and,{'section': '4. Service Levels'}
4,2aac5c88-747c-4e5a-aa4a-bb662b8bed8f,"384 numbers: [0.02537059411406517, 0.032088316...",4.2 and shall be delivered within 10 working d...,{'section': '4. Service Levels'}
5,4af15824-e69d-4b7e-8beb-44f09e8652b4,"384 numbers: [0.038281671702861786, -0.0426363...",4.4 Service levels are measured per calendar m...,{'section': '4. Service Levels'}
6,cc6df928-13fd-491f-88f8-087cbe47b793,"384 numbers: [-0.00774477981030941, -0.0132773...",4.5 SwiftShip shall share a monthly performanc...,{'section': '4. Service Levels'}
7,7bb03587-3f0a-406e-94c4-26554e605d8d,"384 numbers: [-0.00695929117500782, 0.00533153...","4.6 During the festive peak period, defined as...",{'section': '4. Service Levels'}
8,f280fa85-05bb-4c8f-bb8f-bcf550428263,"384 numbers: [-0.05399357154965401, -0.0891747...",5.1 If SwiftShip misses the metro service leve...,{'section': '5. Service Credits and Penalties'}
9,b44c1aab-5c53-493b-8970-90ec650fc62d,"384 numbers: [0.037666499614715576, 0.02030908...","4.1 in any month, it shall credit ShopEasy 1 p...",{'section': '5. Service Credits and Penalties'}


In [ ]:
question = "SwiftShip delivered 92 percent of metro orders on time in March. How much credit does ShopEasy get, and is it paid in cash?"

top_clause = child_store.similarity_search(question, k=1)[0]           # 1. vector search on clauses
section_name = top_clause.metadata["section"]                            # 2. which section is it from?
parent = db.execute("SELECT text FROM parents WHERE section = ?",       # 3. fetch the whole section
                    (section_name,)).fetchone()[0]

print("Matched clause:", top_clause.page_content)
print("Section:", section_name)
print("\nParent from SQLite:\n", parent)

Matched clause: 5.5 If SwiftShip misses the metro service level for three consecutive months, ShopEasy may terminate this Agreement under Section 9.
Section: 5. Service Credits and Penalties

Parent from SQLite:
 5. Service Credits and Penalties
5.1 If SwiftShip misses the metro service level in 4.1 in any month, it shall credit ShopEasy 1 percent of that month's invoice for every full percentage point below 95 percent. 5.2 If SwiftShip misses the non-metro service level in 4.2, the credit is 0.5 percent of the monthly invoice for every full percentage point below 90 percent. 5.3 Total service credits in any month shall not exceed 10 percent of that month's invoice. 5.4 Service credits are adjusted against the next invoice and are not paid in cash. 5.5 If SwiftShip misses the metro service level for three consecutive months, ShopEasy may terminate this Agreement under Section 9. 5.6 Service credits are ShopEasy's only financial remedy for delays, except where the delay was caused by gr

In [ ]:
question = "SwiftShip delivered 92 percent of metro orders on time in March. How much credit does ShopEasy get, and is it paid in cash?"

clauses = child_store.similarity_search(question, k=2)
without_parent = "\n".join(c.page_content for c in clauses)              # only the matched clauses
with_parent = parent                                                     # the whole section from SQLite

for name, context in [("Without parent (clauses only)", without_parent), ("With parent (whole section)", with_parent)]:
    response = llm.invoke([
        ("system", f"""You are ShopEasy's contracts assistant.
Answer only using the contract text below. If the answer is not there, say you don't know.

<contract>
{context}
</contract>"""),
        ("user", question),
    ])
    display(Markdown(f"### {name}\n**Context sent:**\n\n{context}\n\n**Answer:** {response.content}"))

### Without parent (clauses only)
**Context sent:**

5.5 If SwiftShip misses the metro service level for three consecutive months, ShopEasy may terminate this Agreement under Section 9.
5.2 If SwiftShip misses the non-metro service level in 4.2, the credit is

**Answer:** I don't know.

### With parent (whole section)
**Context sent:**

5. Service Credits and Penalties
5.1 If SwiftShip misses the metro service level in 4.1 in any month, it shall credit ShopEasy 1 percent of that month's invoice for every full percentage point below 95 percent. 5.2 If SwiftShip misses the non-metro service level in 4.2, the credit is 0.5 percent of the monthly invoice for every full percentage point below 90 percent. 5.3 Total service credits in any month shall not exceed 10 percent of that month's invoice. 5.4 Service credits are adjusted against the next invoice and are not paid in cash. 5.5 If SwiftShip misses the metro service level for three consecutive months, ShopEasy may terminate this Agreement under Section 9. 5.6 Service credits are ShopEasy's only financial remedy for delays, except where the delay was caused by gross negligence.

**Answer:** ShopEasy receives a credit of **3 percent of March’s invoice** (the service was 3 full percentage points below the 95 percent metro level, and the credit is 1 percent per point).  

The credit is **not paid in cash**; it is applied as an adjustment against the next invoice.

> **Takeaway:** Same question, same search. The clauses alone matched the topic but missed the rule (5.1) and the payment term (5.4), so the model correctly said "I don't know". The parent section had both. Search small to find the right place; send big so the model can answer.

### How big should a chunk be?
Same splitter (recursive), two real questions, different chunk sizes.

In [ ]:
questions = {
    "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?": "within 15 days",
    "How much does ShopEasy Plus membership cost?": "Rs 499 per year",
}

for size in [100, 200, 500, 1000, 2000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10)
    store = InMemoryVectorStore.from_documents(splitter.create_documents([handbook]), embeddings)

    print(f"===== chunk_size {size} =====")
    for question in questions:
        context = "\n\n".join(doc.page_content for doc in store.similarity_search(question, k=3))
        response = llm.invoke([
            ("system", f"""You are ShopEasy's customer support assistant.
Answer only using the context below. If the answer is not there, say you don't know.

<context>
{context}
</context>"""),
            ("user", question),
        ])
        print(f"Q: {question}")
        print(f"A: {response.content}\n")
    print()

===== chunk_size 100 =====
Q: My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?
A: Yes—you can request a free size exchange for your shoes as long as you’re still within the return window. Since you received them 20 days ago, you’re still within that period. Just let us know the new size you need; we’ll arrange for the original pair to be picked up and deliver the replacement size to you (subject to stock availability).

Q: How much does ShopEasy Plus membership cost?
A: ShopEasy Plus membership costs Rs 499 per year.


===== chunk_size 200 =====
Q: My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?
A: Yes—you can request a free size exchange for your shoes because you’re still within the return window. We’ll arrange for the original pair to be picked up, and once the new size is in stock we’ll ship it to you. If the larger size isn’t available, we’ll process a refund instead.

Q: How much 

| chunk_size | Shoes (correct: **No**, 15 days) | Plus price (correct: **Rs 499**) |
|---|---|---|
| 100 | ❌ "Yes, you're likely still eligible" | ✅ |
| 200 | ❌ "Yes" | ✅ |
| 500 | ⚠️ Hedges: "if 20 days is past the window…" (never saw the 15 days) | ✅ |
| **1000** | ✅ "No, 15 days" | ✅ |
| 2000 | ✅ "No, 15 days" | ❌ "I don't know" |

> **Takeaway:** Too small (100–500): the shoes rule got split across chunks and the model guessed "Yes". Too big (2000): the Plus price got buried in a giant mixed chunk and the model said "I don't know". Only 1000 answered both correctly. Chunk size is a trade-off, and the right value depends on your data and questions, so test it.

In [ ]:
tests = {
    "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?": "within 15 days",
    "How much does ShopEasy Plus membership cost?": "Rs 499 per year",
    "What is the customer care phone number?": "1800-000-4321",
    "Can I pay cash on delivery for a Rs 15,000 order?": "up to Rs 10,000",
    "Can I return a perfume?": "perfumes and personalised items cannot be returned",
    "Can I buy ShopEasy Care for my laptop after 2 months?": "within 30 days of delivery",
}


In [ ]:
for size in [100, 200, 500, 1000, 2000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 10)
    store = InMemoryVectorStore.from_documents(splitter.create_documents([handbook]), embeddings)

    score, chars = 0, 0
    for question, answer in tests.items():
        context = " ".join(doc.page_content for doc in store.similarity_search(question, k=3))
        chars += len(context)
        if answer in context:
            score += 1
    print(f"size {size:5} → found {score}/6, avg characters sent {chars // 6}")

size   100 → found 4/6, avg characters sent 254
size   200 → found 5/6, avg characters sent 399
size   500 → found 5/6, avg characters sent 1150
size  1000 → found 6/6, avg characters sent 2353
size  2000 → found 5/6, avg characters sent 4731


> **Takeaway:** Bigger chunks find more answers, up to a point: 1000 found all 6. At 2000 the chunks mix so many topics that the Plus price got lost, and we send 4,700 characters per question, twice as much as 1000 and 4× as much as 500. Pick the smallest size that still finds your answers.

### Why chunk overlap?
When a chunk boundary falls in the middle of a fact, that fact gets split between two chunks, and neither chunk looks relevant on its own. Overlap repeats the end of each chunk at the start of the next, so the fact appears whole at least once.

In [ ]:
for overlap in [0, 50, 100]:
    splitter = CharacterTextSplitter(separator="", chunk_size=300, chunk_overlap=overlap)
    store = InMemoryVectorStore.from_documents(splitter.create_documents([handbook]), embeddings)

    score = 0
    for question, answer in tests.items():
        context = " ".join(doc.page_content for doc in store.similarity_search(question, k=3))
        if answer in context:
            score += 1
    print(f"overlap {overlap:3} → found {score}/6")

overlap   0 → found 3/6
overlap  50 → found 4/6
overlap 100 → found 5/6


> **Takeaway:** More overlap, more answers found: 3/6 → 4/6 → 5/6. Facts that fell on a chunk boundary (like the phone number) now appear whole in at least one chunk. The cost is more chunks to store and embed (38 → 56). Overlap reduces boundary losses; it can't fix a rule split across sections, which is why the shoes question still fails at every setting.

---
## RAG as a Retrieval tool


In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def search_handbook(query: str) -> str:
    """Search ShopEasy's customer policy handbook for returns, refunds, delivery, payments and contact details."""
    results = section_store.similarity_search(query, k=3)
    return "\n\n".join(doc.page_content for doc in results)

agent = create_agent(
    model=llm,
    tools=[search_handbook],
    system_prompt="You are ShopEasy's customer support assistant. If you can't find the answer, say you don't know.",
)

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?"}]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

My new running shoes are too small. I got them 20 days ago. Can I exchange them for a bigger size?
================================== Ai Message ==================================
Tool Calls:
  search_handbook (fc_1fabd0ea-a376-4c86-aa16-5702a98a5710)
 Call ID: fc_1fabd0ea-a376-4c86-aa16-5702a98a5710
  Args:
    query: exchange shoes size too small 20 days policy
================================= Tool Message =================================
Name: search_handbook

9. Size Exchange
For clothing and footwear, you can request one free size exchange within the return window. The new size is delivered when the original item is picked up. Size exchange is subject to stock availability. If the requested size is out of stock, a refund is processed instead.

8. Returns Policy
Dresses, suits and party wear can be returned within 7 days of delivery. All other clothing and footwear can be returned within 15 days of 

### What happens inside the agent: 2 LLM calls

**Call 1: decide**
- Input: system prompt + tool definition + customer question
- Output: a tool call → `search_handbook("exchange policy size too small 20 days shoes")`

*Our code runs the tool and gets the 3 sections.*

**Call 2: answer**
- Input: everything from call 1 + the tool call + the retrieved sections
- Output: the final answer

> **No memory needed here:** both calls happen inside one `agent.invoke()`, and the agent re-sends the full message list each time. Memory matters only across separate questions.

---
## Code needs code-aware chunking
Code has structure too: classes and functions. Cutting it every N characters breaks functions in the middle.

In [ ]:
code = '''class ReturnService:
    """Handles customer returns."""

    def is_returnable(self, category, days_since_delivery):
        windows = {"dress": 7, "footwear": 15, "home": 10}
        if category in ("innerwear", "perfume"):
            return False
        return days_since_delivery <= windows.get(category, 15)

    def schedule_pickup(self, order_id):
        return f"Pickup for {order_id} within 2 working days"


class RefundService:
    """Handles refunds after quality check."""

    def refund_days(self, payment_method):
        if payment_method == "UPI":
            return "2 to 3 working days"
        if payment_method == "COD":
            return "7 to 10 working days"
        return "5 to 7 working days"
'''

In [ ]:
from langchain_text_splitters import Language

print("===== Fixed-size =====")
for chunk in CharacterTextSplitter(separator="", chunk_size=200, chunk_overlap=0).split_text(code):
    print(chunk, "\n-----")

print("\n===== Code-aware (Python) =====")
python_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, chunk_size=300, chunk_overlap=0)
for chunk in python_splitter.split_text(code):
    print(chunk, "\n-----")

===== Fixed-size =====
class ReturnService:
    """Handles customer returns."""

    def is_returnable(self, category, days_since_delivery):
        windows = {"dress": 7, "footwear": 15, "home": 10}
        if category in 
-----
("innerwear", "perfume"):
            return False
        return days_since_delivery <= windows.get(category, 15)

    def schedule_pickup(self, order_id):
        return f"Pickup for {order_id} with 
-----
in 2 working days"


class RefundService:
    """Handles refunds after quality check."""

    def refund_days(self, payment_method):
        if payment_method == "UPI":
            return "2 to 3 work 
-----
ing days"
        if payment_method == "COD":
            return "7 to 10 working days"
        return "5 to 7 working days" 
-----

===== Code-aware (Python) =====
class ReturnService:
    """Handles customer returns.""" 
-----
def is_returnable(self, category, days_since_delivery):
        windows = {"dress": 7, "footwear": 15, "home": 10}
        if

In [ ]:
code_chunks = python_splitter.create_documents([code])
code_store = InMemoryVectorStore.from_documents(code_chunks, embeddings)

In [ ]:
question = "Which function decides whether an item can be returned, and what is the return window for footwear?"

results = code_store.similarity_search(question, k=2)
for doc in results:
    print(doc.page_content, "\n-----")

context = "\n\n".join(doc.page_content for doc in results)

response = llm.invoke([
    ("system", f"""You are a senior engineer at ShopEasy. Answer questions about the codebase using only the code below. If the answer is not there, say you don't know.

<code>
{context}
</code>"""),
    ("user", question),
])

print(response.content)

def is_returnable(self, category, days_since_delivery):
        windows = {"dress": 7, "footwear": 15, "home": 10}
        if category in ("innerwear", "perfume"):
            return False
        return days_since_delivery <= windows.get(category, 15) 
-----
class RefundService:
    """Handles refunds after quality check.""" 
-----
The function that determines if an item can be returned is **`is_returnable`**.

For the **`footwear`** category, the return window is **15 days** (as defined in the `windows` dictionary).


In [ ]:
tagged_code_chunks = []
current_class = None

for piece in python_splitter.split_text(code):
    if piece.startswith("class "):                       # a new class starts here
        current_class = piece.split()[1].rstrip(":")      # "class RefundService:" → "RefundService"
    tagged_code_chunks.append(Document(
        page_content=f"# class: {current_class}\n{piece}",
        metadata={"class": current_class},
    ))

tagged_code_store = InMemoryVectorStore.from_documents(tagged_code_chunks, embeddings)  # <-- add this line

for chunk in tagged_code_chunks:
    print(chunk.page_content, "\n-----")

# class: ReturnService
class ReturnService:
    """Handles customer returns.""" 
-----
# class: ReturnService
def is_returnable(self, category, days_since_delivery):
        windows = {"dress": 7, "footwear": 15, "home": 10}
        if category in ("innerwear", "perfume"):
            return False
        return days_since_delivery <= windows.get(category, 15) 
-----
# class: ReturnService
def schedule_pickup(self, order_id):
        return f"Pickup for {order_id} within 2 working days" 
-----
# class: RefundService
class RefundService:
    """Handles refunds after quality check.""" 
-----
# class: RefundService
def refund_days(self, payment_method):
        if payment_method == "UPI":
            return "2 to 3 working days"
        if payment_method == "COD":
            return "7 to 10 working days"
        return "5 to 7 working days" 
-----


In [ ]:
retrieval_query = "How long does a UPI refund take?"
question = "Which class handles this, and how long does a UPI refund take?"

for name, store in [("Plain (no tag)", code_store), ("Tagged", tagged_code_store)]:
    context = "\n\n".join(doc.page_content for doc in store.similarity_search(retrieval_query, k=1))
    print(f"===== {name} =====")
    print("Context retrieved:", context, "\n")

    response = llm.invoke([
        ("system", f"""You are a senior engineer at ShopEasy. Answer questions about the codebase using only the code below. If the answer is not there, say you don't know.

<code>
{context}
</code>"""),
        ("user", question),
    ])
    print("A:", response.content, "\n")

===== Plain (no tag) =====
Context retrieved: class RefundService:
    """Handles refunds after quality check.""" 

A: The **`RefundService`** class handles refunds.

The code you provided does not contain any information about how long a UPI refund takes, so I don’t know. 

===== Tagged =====
Context retrieved: # class: RefundService
def refund_days(self, payment_method):
        if payment_method == "UPI":
            return "2 to 3 working days"
        if payment_method == "COD":
            return "7 to 10 working days"
        return "5 to 7 working days" 

A: The **`RefundService`** class handles this logic.  
For a payment method of **UPI**, the `refund_days` method returns **“2 to 3 working days.”** 

